Lma t5alasy remove print statements

# BloomBERT — DistilBERT fine-tuned for Bloom-level (3-class) classification

Trains a DistilBERT classifier that takes a question and predicts difficulty: `easy` / `medium` / `hard`.

## Techniques applied
- **Frozen bottom 4 transformer layers** — keep general syntax representations stable; only top 2 adapt to Bloom semantics
- **Class-weighted cross-entropy** — corrects the 65/25/10 class imbalance
- **fp16 mixed precision (AMP)** — ~2x speed-up on T4, identical accuracy
- **Layer-wise learning rate decay (LLRD)** — top layers learn faster than bottom layers; documented +0.5–1% F1 for BERT family
- **Linear warmup + linear decay** — standard BERT fine-tuning schedule
- **Best-checkpoint selection by val macro-F1** — robust to occasional bad final epoch

## Setup
1. Upload `train.csv`, `val.csv`, `test.csv` (from `data/processed/`) as a **private Kaggle dataset** called e.g. `baylearn-bloom-data`.
2. Add that dataset to this notebook (right-side panel → Add Input → your dataset).
3. Enable **GPU T4** in notebook settings (Accelerator → GPU T4 x1).
4. Run all cells. Total runtime ~15–25 min on T4 with fp16.
5. The trained model is saved under `/kaggle/working/bloom_distilbert/`. Download the whole folder via the right-side panel.

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
# autocast automatically chooses (fp16 or fp32) for GPU operations to improve performance and reduce memory usage
# GradScaler helps prevent underflow in gradients when using mixed precision training by dynamically scaling the loss value
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    DistilBertTokenizerFast, # tokenizer transforms text into numbers that the model can understand
    DistilBertForSequenceClassification, # ready model with a classification head on top
    get_linear_schedule_with_warmup, # learning rate scheduler that increases the learning rate linearly during a warmup period and then decreases it linearly to zero over the remaining training steps
)
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Config = {
    'model_name'   : 'distilbert-base-uncased',
    'max_len'      : 128,
    'batch_size'   : 32,
    'epochs'       : 4,
    'lr'           : 2e-5,           # top-layer learning rate
    'llrd_decay'   : 0.9,            # each lower layer gets lr * 0.9^depth b7es tt3lm 2bt2 l2nha mt3lema kwayes fe el features bto3ha
    'weight_decay' : 0.01,           # L2 regularization to prevent overfitting by penalizing large weights
    'warmup_frac'  : 0.10,           # first 10% of training steps the learning rate is increased linearly from 0 to the initial learning rate, then decayed linearly to 0 over the remaining steps
    'unfreeze_top' : 3,              # best from ablation (1: 0.7051, 2: 0.7170, 3: 0.7261)
    'num_classes'  : 3,
    'use_amp'      : True,           # fp16 mixed precision
}
LABEL_TO_ID = {'easy': 0, 'medium': 1, 'hard': 2}
ID_TO_LABEL = {v: k for k, v in LABEL_TO_ID.items()}

DATA_DIR = '/kaggle/input/datasets/manarabdelshafy/baylearn-bloom-data'
OUT_DIR  = '/kaggle/working/bloom_distilbert'
os.makedirs(OUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('fp16 available:', Config['use_amp'])

In [ ]:
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
val_df   = pd.read_csv(f'{DATA_DIR}/val.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    before = len(df)
    df.dropna(subset=['question', 'level'], inplace=True)
    df['level'] = df['level'].str.lower().str.strip()
    df.drop(df[~df['level'].isin(LABEL_TO_ID.keys())].index, inplace=True)
    print(f'  {name}: {len(df)}/{before} after cleaning')

print('\nLabel distribution in train:')
print(train_df['level'].value_counts())
print('\nSource distribution in train:')
if 'source' in train_df.columns:
    print(train_df['source'].value_counts())

In [ ]:
# load the tokenizer for the specified model, which will convert text into token IDs identified in the model corpus that the model can process 
tokenizer = DistilBertTokenizerFast.from_pretrained(Config['model_name'])

class BloomDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts  = df['question'].astype(str).tolist()
        self.labels = [LABEL_TO_ID[l] for l in df['level']]
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0), # when we pad to max_length, we need to tell the model which tokens are padding and which are actual data using the attention mask so they are [1, 1, 1, 0, 0] for example
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = BloomDataset(train_df, tokenizer, Config['max_len'])
val_ds   = BloomDataset(val_df,   tokenizer, Config['max_len'])
test_ds  = BloomDataset(test_df,  tokenizer, Config['max_len'])
# pin memory=True allows faster data transfer to GPU by keeping the data in page-locked memory, which can speed up training when using a GPU
# shuffle 34an el data tb2a mtnw3a m4 easy easy medium and next batch medium medium hard hard 
train_loader = DataLoader(train_ds, batch_size=Config['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=Config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=Config['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

# check tokenized lengths to see if we can reduce max_len to speed up training and save memory not to save 128 and max question is only 60 tokens
lengths = train_df['question'].astype(str).apply(lambda s: len(tokenizer.tokenize(s)))
print(f'Tokenized length: median={int(lengths.median())}  p95={int(lengths.quantile(0.95))}  max={int(lengths.max())}')
if lengths.quantile(0.95) < 64:
    print(f'>>> 95th percentile is under 64 tokens — consider Config["max_len"]=64 to halve training time.')

In [ ]:
def build_model(unfreeze_top: int):
    """distilbert has 6 layers we only unfreeze last layers to make the task specific for our goal"""
    model = DistilBertForSequenceClassification.from_pretrained(
        Config['model_name'], num_labels=Config['num_classes']
    )
    for p in model.parameters():
        p.requires_grad = False # freeze all layers first
    layers = model.distilbert.transformer.layer  # 6 TransformerBlocks
    for i in range(6 - unfreeze_top, 6):
        for p in layers[i].parameters():
            p.requires_grad = True # unfreeze the last 'unfreeze_top' layers
    for p in model.pre_classifier.parameters(): p.requires_grad = True # the pre-classifier layer that comes after the transformer layers should also be trainable
    for p in model.classifier.parameters():     p.requires_grad = True # the classification head on top should always be trainable
    return model.to(device)

_m = build_model(Config['unfreeze_top'])
total = sum(p.numel() for p in _m.parameters()) # total number of parameters in the model, which is the sum of the number of elements in each parameter tensor
train_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'Total params: {total:,}   Trainable: {train_p:,}   ({100*train_p/total:.1f}%)')
del _m; torch.cuda.empty_cache() # free up memory after the parameter count check to avoid OOM during training

In [ ]:
counts = train_df['level'].value_counts()
total = counts.sum()
weights = {level: total / (Config['num_classes'] * counts[level]) for level in counts.index}
class_weights = torch.tensor(
    [weights[ID_TO_LABEL[i]] for i in range(Config['num_classes'])],
    dtype=torch.float, device=device,
)
print('Class weights:', {ID_TO_LABEL[i]: round(class_weights[i].item(), 3) for i in range(3)})

In [ ]:
def build_llrd_parameter_groups(model, base_lr: float, decay: float, weight_decay: float):
    """
Embeddings
|
Transformer Layer 0
|
Transformer Layer 1
|
Transformer Layer 2
|
Transformer Layer 3
|
Transformer Layer 4
|
Transformer Layer 5
| those next layers are added by HFACE to make the language model do the classification task
they are grouped after doing the LLRD for normal layers we donot do any decay for them 
pre_classifier  just processing 
|
classifier  heads 
We use AdamW for updating gradients as it is the optimised way for handling weights in backward propagation
    """
    no_decay_names = ('bias', 'LayerNorm.weight', 'LayerNorm.bias')
    # we do not apply weight decay for them because bias is not a cause for complexity it
    # just shifts the function and LayerNorm  it is responsible for normalizing the weights 
    # so regulazing it causes the normalization to be weaker and unstable
    groups = []
    # 6 layers so layer 5 = depth 0 (top), layer 0 = depth 5 (bottom)
    for layer_idx in range(6):
        depth_from_top = 5 - layer_idx
        lr = base_lr * (decay ** depth_from_top)
        # because when we are looping on parameters we can check if this paramater belongs to this layer or not so we skip it if not
        prefix = f'distilbert.transformer.layer.{layer_idx}.'
        decay_params, nodecay_params = [], []
        for n, p in model.named_parameters():
            if not p.requires_grad or not n.startswith(prefix):
                continue
            if any(nd in n for nd in no_decay_names):
                nodecay_params.append(p)
            else:
                decay_params.append(p)
        # now the optimizer we deal differently with each layer parameters according to its depth(LR is changed according to their depth)
        if decay_params:
            groups.append({'params': decay_params, 'lr': lr, 'weight_decay': weight_decay})
        # no weight decay for those 
        if nodecay_params:
            groups.append({'params': nodecay_params, 'lr': lr, 'weight_decay': 0.0})
            
    # those are the new heads layer (easy - medium - hard) so apply for them the full learning rate as they are new 
    # Head (pre_classifier + classifier) gets base_lr
    head_decay, head_nodecay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if 'classifier' not in n and 'pre_classifier' not in n:
            continue
        if any(nd in n for nd in no_decay_names):
            head_nodecay.append(p)
        else:
            head_decay.append(p)
    if head_decay:
        groups.append({'params': head_decay, 'lr': base_lr, 'weight_decay': weight_decay})
    if head_nodecay:
        groups.append({'params': head_nodecay, 'lr': base_lr, 'weight_decay': 0.0})
    # we return at the end all these groups we gathered to give to the optimizer 
    return groups

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, scaler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        # non blocking to make work asynchrounous not wait for cpu and will not affect if pin memory and using gpu didnot happen 
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attn_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels    = batch['labels'].to(device, non_blocking=True) 
        if Config['use_amp']:
            with autocast():
                out  = model(input_ids=input_ids, attention_mask=attn_mask)
                loss = loss_fn(out.logits, labels)
            scaler.scale(loss).backward() # we did loss * scale to prevent underflow in gradients when using amp
            scaler.unscale_(optimizer) # return gradients to their original scale before clipping to prevent gradient explosion in amp
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)# if gradient exploding happened minimize gradient 
            scaler.step(optimizer)  # updating weights
            scaler.update() 
        else:
            out  = model(input_ids=input_ids, attention_mask=attn_mask)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        scheduler.step() # changing LR according to the scheduler after each batch
        total_loss += loss.item()
    return total_loss / len(loader)
# we say here do not record gradients we are not traning  (do not record computation graph) we said that by putting a decorator for the whole
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    preds, gold = [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attn_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels    = batch['labels'].to(device, non_blocking=True)
        if Config['use_amp']: # if you uses Automatic Mixed Precision which transform f32 to f16
            with autocast():
                out = model(input_ids=input_ids, attention_mask=attn_mask)
        else:
            out = model(input_ids=input_ids, attention_mask=attn_mask)
        total_loss += loss_fn(out.logits.float(), labels).item()
        preds.extend(out.logits.argmax(dim=-1).cpu().tolist()) # logits are numbers output
        gold.extend(labels.cpu().tolist())
    macro = f1_score(gold, preds, average='macro')
    return total_loss / len(loader), macro, preds, gold

In [ ]:
def save_test_artifacts(out_dir: str, preds, gold, test_loss, test_f1,
                        history, config, unfreeze_top, df_for_questions=None,
                        prefix: str = ""):
    """Save everything from a test run that you'd want in your report.

    Files written (under `out_dir`):
      - {prefix}training_history.json       (per-epoch train/val loss + F1)
      - {prefix}test_classification_report.txt  (sklearn report, human-readable)
      - {prefix}test_classification_report.json (same numbers as JSON for programs)
      - {prefix}test_confusion_matrix.csv   (rows=true, cols=pred, easy/medium/hard)
      - {prefix}test_predictions.csv        (per-question: question, true, pred,
                                             correct, optional source — for error
                                             analysis)
      - {prefix}label_map.json              (label string ↔ int ID mapping)
    """
    import csv
    os.makedirs(out_dir, exist_ok=True)
    target_names = ['easy', 'medium', 'hard']

    # 1. Training history (per-epoch curves)
    with open(f'{out_dir}/{prefix}training_history.json', 'w') as f:
        json.dump({
            'history': history,
            'test_macro_f1': test_f1,
            'test_loss': test_loss,
            'config': config,
            'unfreeze_top': unfreeze_top,
        }, f, indent=2)

    # 2. Classification report (human + machine readable)
    text_report = classification_report(gold, preds, target_names=target_names, digits=4)
    with open(f'{out_dir}/{prefix}test_classification_report.txt', 'w') as f:
        f.write(f'TEST macro_f1 = {test_f1:.4f}   loss = {test_loss:.4f}\n\n')
        f.write(text_report)
    dict_report = classification_report(gold, preds, target_names=target_names,
                                        digits=4, output_dict=True)
    with open(f'{out_dir}/{prefix}test_classification_report.json', 'w') as f:
        json.dump(dict_report, f, indent=2)

    # 3. Confusion matrix as CSV (rows=true, cols=pred)
    cm = confusion_matrix(gold, preds)
    with open(f'{out_dir}/{prefix}test_confusion_matrix.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['true \\ pred'] + target_names)
        for i, row in enumerate(cm):
            w.writerow([target_names[i]] + list(row))

    # 4. Per-question predictions (for error analysis)
    with open(f'{out_dir}/{prefix}test_predictions.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['idx', 'question', 'true_level', 'predicted_level', 'correct', 'source'])
        questions = df_for_questions['question'].tolist() if df_for_questions is not None else [''] * len(gold)
        sources   = df_for_questions['source'].tolist()   if (df_for_questions is not None and 'source' in df_for_questions.columns) else [''] * len(gold)
        for i, (q, t, p) in enumerate(zip(questions, gold, preds)):
            w.writerow([i, q, target_names[t], target_names[p],
                        'yes' if t == p else 'no',
                        sources[i] if i < len(sources) else ''])

    # 5. Label map
    with open(f'{out_dir}/{prefix}label_map.json', 'w') as f:
        json.dump(LABEL_TO_ID, f, indent=2)


def run_training(unfreeze_top: int, save: bool = True):
    model = build_model(unfreeze_top)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)
    param_groups = build_llrd_parameter_groups(
        model, base_lr=Config['lr'], decay=Config['llrd_decay'],
        weight_decay=Config['weight_decay'],
    )
    optimizer = torch.optim.AdamW(param_groups)
    total_steps = len(train_loader) * Config['epochs']
    warmup_steps = int(total_steps * Config['warmup_frac'])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps,
    )
    scaler = GradScaler(enabled=Config['use_amp'])
    history = []
    best_f1, best_state = -1.0, None
    for epoch in range(1, Config['epochs'] + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler)
        val_loss, val_f1, _, _ = evaluate(model, val_loader, loss_fn)
        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_loss': val_loss, 'val_macro_f1': val_f1})
        print(f'  epoch {epoch}/{Config["epochs"]}   '
              f'train_loss={train_loss:.4f}   val_loss={val_loss:.4f}   '
              f'val_macro_f1={val_f1:.4f}')
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    test_loss, test_f1, preds, gold = evaluate(model, test_loader, loss_fn)
    print(f'\nTEST  macro_f1={test_f1:.4f}   loss={test_loss:.4f}')
    print('\n' + classification_report(gold, preds, target_names=['easy', 'medium', 'hard'], digits=4))
    print('Confusion matrix (rows=true, cols=pred):  easy / medium / hard')
    print(confusion_matrix(gold, preds))
    if save:
        model.save_pretrained(OUT_DIR)
        tokenizer.save_pretrained(OUT_DIR)
        save_test_artifacts(
            out_dir=OUT_DIR, preds=preds, gold=gold,
            test_loss=test_loss, test_f1=test_f1, history=history,
            config=Config, unfreeze_top=unfreeze_top,
            df_for_questions=test_df,
        )
        print(f'\nSaved model + tokenizer + report artifacts to {OUT_DIR}')
        print(f'  - model.safetensors / config.json / tokenizer files')
        print(f'  - training_history.json')
        print(f'  - test_classification_report.txt + .json')
        print(f'  - test_confusion_matrix.csv')
        print(f'  - test_predictions.csv')
    return test_f1, history

print(f'\n=== Main run: unfreezing top {Config["unfreeze_top"]} layers ===')
main_f1, main_history = run_training(Config['unfreeze_top'], save=True)

In [ ]:
# Ablation — unfreeze 1 or 2 or 3 layers 
ablation_results = {Config['unfreeze_top']: main_f1}
for ut in [1, 2, 3]:
    if ut == Config['unfreeze_top']:
        continue
    print(f'\n=== Ablation: unfreezing top {ut} layer(s) ===')
    f1, _ = run_training(ut, save=False)
    ablation_results[ut] = f1

print('\n\n=== Ablation summary ===')
print('Unfrozen top layers → test macro-F1')
for ut in sorted(ablation_results.keys()):
    print(f'  {ut} layer(s):  {ablation_results[ut]:.4f}')
with open(f'{OUT_DIR}/ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=2)

In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
loaded_model = DistilBertForSequenceClassification.from_pretrained(OUT_DIR).to(device).eval()
loaded_tok = DistilBertTokenizerFast.from_pretrained(OUT_DIR)

samples = [
    'Define a semaphore.',
    'Compute the number of page faults for the reference string 1,2,3,4,1,2,5 under LRU with 3 frames.',
    'Design a deadlock-free dining philosophers protocol using only binary semaphores and justify why it avoids circular wait.',
]
with torch.no_grad():
    enc = loaded_tok(samples, padding=True, truncation=True,
                     max_length=Config['max_len'], return_tensors='pt').to(device)
    logits = loaded_model(**enc).logits
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = logits.argmax(dim=-1).cpu().tolist()
for s, p, pr in zip(samples, preds, probs):
    print(f'\n[{ID_TO_LABEL[p]}]  (easy={pr[0]:.2f}  medium={pr[1]:.2f}  hard={pr[2]:.2f})')
    print(f'  Q: {s}')

In [ ]:
# ---- Final cell: ZIP the output for reliable single-file download -------
# Kaggle's per-file browser download is flaky on large files (>200 MB).
# Solution: archive the entire OUT_DIR into one zip and download that.

import shutil, os
zip_path = shutil.make_archive('/kaggle/working/bloom_distilbert', 'zip', OUT_DIR)
sz_mb = os.path.getsize(zip_path) / 1e6
print(f'Created: {zip_path}  ({sz_mb:.1f} MB)')
print(f'\nDownload steps:')
print(f'  1. Open the right-side "Output" panel.')
print(f'  2. Find: bloom_distilbert.zip')
print(f'  3. Click the download icon next to it.')
print(f'  4. Locally: unzip to question-generation-module/models/bloom_distilbert/')